# NBA Salary Prediction: A Fair-Market Value Detector

**IEOR 142A — Spring 2026 Project**

---

## Abstract

NBA player salaries are determined by a mix of on-court production, market dynamics, and Collective Bargaining Agreement (CBA) constraints — rookie-scale contracts, max-salary tiers, supermax eligibility, and Bird rights all decouple pay from pure performance. This project builds a fair-market value model that predicts a player's 2025-26 salary from their 2024-25 production, then uses out-of-fold residuals to flag the league's most over- and underpaid contracts.

We compare five regression methods (Linear, Ridge, Lasso, Random Forest, XGBoost) using 5-fold cross-validation on a feature-engineered dataset of 344 above-league-minimum NBA players. **XGBoost achieves the best cross-validated performance (CV R² ≈ 0.55)**, with the spread between models being small enough that the underlying signal — about 50–55% of salary variance is explainable from one season of stats — is roughly stable across model classes. The remaining variance comes from precisely the CBA-induced effects we expected: recently extended young stars producing below their fresh-extension price (Suggs, LaMelo, Green) appear systematically "overpaid," while ascendant rookie-scale players (Jalen Williams, Reaves, Avdija) and pre-MVP-extension stars (SGA) appear "underpaid."

The model is deployed as an interactive Streamlit dashboard for predicting fair-market value, exploring the league-wide leaderboard, and decomposing individual predictions via SHAP.


## ⚙️ Colab setup (skip if running locally)

If you're running this in Google Colab, run the cell below first to install missing packages and upload the two CSVs. If running locally with a normal Python env, skip past it.

In [ ]:
# Colab setup — only run this in Colab. Skip if running locally.
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # 1. Install missing packages
    !pip install -q xgboost shap

    # 2. Upload the two CSVs (NBA2024-2025.csv and NBA2025Salary.csv)
    from google.colab import files
    print('Upload NBA2024-2025.csv and NBA2025Salary.csv:')
    uploaded = files.upload()
    print('Uploaded:', list(uploaded.keys()))
else:
    print('Local run — make sure NBA2024-2025.csv and NBA2025Salary.csv are in the working directory.')

## 1. Use Case & Motivation

**Who is this for?**
1. **NBA front offices** evaluating trade targets — "is this contract movable, or is the player overpaid relative to production?"
2. **Player agents** building negotiation cases — "comparable production justifies $X based on the league's market"
3. **Fans, analysts, and CBA researchers** identifying market inefficiencies

**Why this is interesting:** A naïve "salary = production" model would be uninteresting and largely tautological. The interesting story lies in the *residuals*: NBA salaries are deliberately not a perfect function of production because of CBA structures (rookie scales, max-contract tiers, length-of-service tied minimums). The residuals from a production-based model are therefore a direct measure of CBA-induced market inefficiency. The model isn't useful because it predicts well — **it's useful because *where it predicts poorly* is informative.**

**Decision-making framing:** A front-office analyst could use this tool at the trade deadline to filter the player pool to "underpaid + young + healthy" candidates (high-upside acquisitions) and avoid "overpaid + aging + injury-prone" candidates (negative-value contracts).


## 2. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import (train_test_split, KFold, cross_val_score,
                                     cross_val_predict, GridSearchCV)
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import xgboost as xgb
import shap

np.random.seed(42)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

## 3. Data Loading & Cleaning

**Sources:**
- `NBA2024-2025.csv` — Per-player regular season totals from Basketball-Reference (2024-25 season)
- `NBA2025Salary.csv` — Contract data from Basketball-Reference's salary tables (2025-26 season)

**Cleaning decisions:**
1. Drop the repeating header rows that Basketball-Reference embeds throughout the CSV
2. For traded players (multiple team rows), keep only the combined `2TM` row
3. Fill missing percentages with 0 (these are players with zero attempts, not true missingness)
4. Engineer binary award flags from the `Awards` column (All-Star, MVP candidate, DPOY candidate)
5. Filter to players with ≥20 games — below this cutoff, salary is largely structural (10-day contracts, two-way deals)
6. Inner-merge on player name to salary table


In [ ]:
# Player stats
df = pd.read_csv('NBA2024-2025.csv').dropna(subset=['Rk'])

# Traded players: keep the combined 2TM row only
traded = df[df['Team'] == '2TM']['Player-additional'].unique()
df = df[~((df['Player-additional'].isin(traded)) & (df['Team'] != '2TM'))]

# Fill percentage NaNs (zero attempts → not missing)
for c in ['3P%', 'FT%', '2P%', 'eFG%', 'FG%']:
    df[c] = df[c].fillna(0)

# Award binaries
df['is_allstar'] = df['Awards'].str.contains('AS', na=False).astype(int)
df['is_mvp_candidate'] = df['Awards'].str.contains('MVP', na=False).astype(int)
df['is_dpoy_candidate'] = df['Awards'].str.contains('DPOY', na=False).astype(int)
df = df.drop(columns=['Awards', 'Rk']).reset_index(drop=True)
df = df[df['G'] >= 20]

# Salaries
salary_df = pd.read_csv('NBA2025Salary.csv', header=1)
salary_df.columns = ['Rk', 'Player', 'Team', '2025-26', '2026-27', '2027-28',
                     '2028-29', '2029-30', '2030-31', 'Guaranteed', 'Player-additional']
for col in ['2025-26', '2026-27', '2027-28', '2028-29', '2029-30', '2030-31', 'Guaranteed']:
    salary_df[col] = salary_df[col].replace(r'[\$,]', '', regex=True).astype(float)
salary_keep = salary_df[['Player', 'Player-additional', '2025-26']].dropna(subset=['2025-26'])

df_merged = df.merge(salary_keep, on='Player', how='inner')
print(f'After merge: {df_merged.shape[0]} players')

## 4. Exploratory Data Analysis

The salary distribution is heavily right-skewed — a feature of the NBA labor market where a small number of superstars on max contracts earn disproportionately. We log-transform the target to satisfy the homoscedasticity assumption of linear models and to make regression coefficients interpret as approximate percentage changes.

**Cutoff justification:** Below approximately \$2.3M, salaries cluster heavily around the league minimum (set by years of service in the CBA, not performance). Including these players would have the model learn "service time → minimum" instead of "production → market value." We filter to the above-minimum sample.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(df_merged['2025-26'] / 1e6, bins=50, color='#1d428a', edgecolor='white')
axes[0].set_xlabel('Salary ($M)')
axes[0].set_title('Raw 2025-26 Salary (skewed)')
axes[0].axvline(2.3, color='#c8102e', linestyle='--', label='$2.3M cutoff')
axes[0].legend()

axes[1].hist(np.log(df_merged['2025-26']), bins=50, color='#1d428a', edgecolor='white')
axes[1].set_xlabel('log(Salary)')
axes[1].set_title('Log Salary (right tail tamed, but bimodal)')

df_model = df_merged[df_merged['2025-26'] > 2_300_000].copy()
axes[2].hist(np.log(df_model['2025-26']), bins=40, color='#c8102e', edgecolor='white')
axes[2].set_xlabel('log(Salary)')
axes[2].set_title(f'Log Salary, above-min only (n={len(df_model)})')

plt.tight_layout()
plt.show()
print(f'\nFinal modeling sample: {df_model.shape[0]} players, '
      f'salary range ${df_model["2025-26"].min()/1e6:.1f}M – ${df_model["2025-26"].max()/1e6:.1f}M')

## 5. Feature Engineering

A first-pass model using raw season totals (PTS, AST, TRB) confounds *role* with *productivity* — a starter playing 35 minutes accumulates more counting stats than an equally efficient bench player playing 18, hiding per-minute value. We engineer:

- **Per-36 minute stats** for PTS, AST, TRB, STL, BLK — the NBA-analytics standard for rate-adjusted production
- **Per-game stats** for TOV, MP — workload and ball-handling-error proxies
- **Age² (`Age_sq`)** — captures the non-linear NBA aging curve where peak earnings cluster in the late-20s
- **`shots_per_min`** — true usage proxy combining FGA and FTA (free throws scaled by 0.44, the standard adjustment)
- **`stocks_per36`** — defensive versatility composite (steals + blocks per 36)
- **Position dummies** (PG / SG / SF / PF / C) — reference category C
- **Award binaries** — All-Star, MVP candidate, DPOY candidate; these are leading indicators of next-contract value and CBA tier eligibility


In [ ]:
# Per-36 stats (rate-adjusted)
for stat in ['PTS', 'AST', 'TRB', 'STL', 'BLK']:
    df_model[f'{stat}_per36'] = (df_model[stat] / df_model['MP']) * 36

# Per-game stats
for stat in ['PTS', 'AST', 'TRB', 'TOV']:
    df_model[f'{stat}_pg'] = df_model[stat] / df_model['G']
df_model['MP_pg'] = df_model['MP'] / df_model['G']

# Aging curve
df_model['Age_sq'] = df_model['Age'] ** 2

# Usage and versatility
df_model['shots_per_min'] = (df_model['FGA'] + 0.44 * df_model['FTA']) / df_model['MP']
df_model['stocks_per36'] = df_model['STL_per36'] + df_model['BLK_per36']

# Position dummies (reference: C)
df_model['Pos_clean'] = df_model['Pos'].str.split('-').str[0]
pos_dummies = pd.get_dummies(df_model['Pos_clean'], prefix='Pos', drop_first=True)
df_model = pd.concat([df_model, pos_dummies], axis=1)

# Target
df_model['log_salary'] = np.log(df_model['2025-26'])

feature_cols = (['Age', 'Age_sq', 'G', 'GS', 'MP_pg',
                 'PTS_per36', 'AST_per36', 'TRB_per36', 'STL_per36', 'BLK_per36',
                 'eFG%', 'FT%', '3P%',
                 'shots_per_min', 'stocks_per36', 'TOV_pg',
                 'is_allstar', 'is_mvp_candidate', 'is_dpoy_candidate']
                + list(pos_dummies.columns))

X = df_model[feature_cols].astype(float)
y = df_model['log_salary']
print(f'Feature count: {X.shape[1]}')
print('Features:', feature_cols)

### 5.1 Multicollinearity check

Several features are correlated by construction (PTS_per36 with shots_per_min; stocks with its components). For tree models this is irrelevant; for linear models we apply Ridge/Lasso to handle it gracefully rather than aggressively dropping features.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 9))
corr = X[[c for c in feature_cols if not c.startswith('Pos_')]].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True,
            cbar_kws={'shrink': 0.8}, annot_kws={'size': 7}, ax=ax)
ax.set_title('Feature Correlation Matrix (non-positional features)', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Train/Test Split & Cross-Validation Setup

We hold out 20% of the data as a final test set, and use 5-fold cross-validation on the training set for both hyperparameter tuning and reporting honest performance estimates. The original code's reliance on a single train/test split was a key methodological concern — with n ≈ 70 in the test set, R² estimates can swing ±0.05 between random splits. Cross-validated estimates with their standard deviation are far more trustworthy.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

def report(name, y_true_log, y_pred_log, cv_r2_array):
    rmse_log = np.sqrt(mean_squared_error(y_true_log, y_pred_log))
    r2 = r2_score(y_true_log, y_pred_log)
    y_true_d = np.exp(y_true_log); y_pred_d = np.exp(y_pred_log)
    rmse_d = np.sqrt(mean_squared_error(y_true_d, y_pred_d))
    mae_d = mean_absolute_error(y_true_d, y_pred_d)
    mape = np.mean(np.abs((y_true_d - y_pred_d) / y_true_d)) * 100
    return {
        'Model': name,
        'CV R² (mean ± sd)': f'{cv_r2_array.mean():.3f} ± {cv_r2_array.std():.3f}',
        'Test R²': round(r2, 3),
        'Test RMSE (log)': round(rmse_log, 3),
        'Test RMSE ($M)': round(rmse_d / 1e6, 2),
        'Test MAE ($M)': round(mae_d / 1e6, 2),
        'Test MAPE (%)': round(mape, 1),
    }

results = []

## 7. Model Comparison

We fit five model classes, each with appropriate hyperparameter tuning via grid search, and compare them on cross-validated R². Linear, Ridge, and Lasso use a `StandardScaler` upstream (regularization is scale-sensitive); tree models do not.

### Model 1: Linear Regression (baseline)


In [ ]:
lr_pipe = Pipeline([('scaler', StandardScaler()), ('lr', LinearRegression())])
cv_r2 = cross_val_score(lr_pipe, X_train, y_train, cv=kf, scoring='r2')
lr_pipe.fit(X_train, y_train)
results.append(report('Linear Regression', y_test, lr_pipe.predict(X_test), cv_r2))
print(results[-1])

### Model 2: Ridge Regression (L2 regularization, tuned)

In [ ]:
ridge_pipe = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge())])
ridge_grid = GridSearchCV(ridge_pipe, {'ridge__alpha': [0.01, 0.1, 1, 10, 100]},
                          cv=kf, scoring='r2')
ridge_grid.fit(X_train, y_train)
cv_r2 = cross_val_score(ridge_grid.best_estimator_, X_train, y_train, cv=kf, scoring='r2')
results.append(report(f"Ridge (α={ridge_grid.best_params_['ridge__alpha']})",
                      y_test, ridge_grid.predict(X_test), cv_r2))
print(results[-1])

### Model 3: Lasso Regression (L1 regularization, tuned)

In [ ]:
lasso_pipe = Pipeline([('scaler', StandardScaler()), ('lasso', Lasso(max_iter=20000))])
lasso_grid = GridSearchCV(lasso_pipe, {'lasso__alpha': [0.001, 0.01, 0.05, 0.1, 0.5]},
                          cv=kf, scoring='r2')
lasso_grid.fit(X_train, y_train)
cv_r2 = cross_val_score(lasso_grid.best_estimator_, X_train, y_train, cv=kf, scoring='r2')
results.append(report(f"Lasso (α={lasso_grid.best_params_['lasso__alpha']})",
                      y_test, lasso_grid.predict(X_test), cv_r2))
print(results[-1])

lasso_coefs = pd.Series(lasso_grid.best_estimator_.named_steps['lasso'].coef_, index=feature_cols)
print(f"\nLasso kept {(lasso_coefs != 0).sum()} of {len(feature_cols)} features.")
print('Top non-zero features by |coefficient|:')
print(lasso_coefs[lasso_coefs != 0].abs().sort_values(ascending=False).head(10))

### Model 4: Random Forest (tuned)

In [ ]:
rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    {'n_estimators': [200, 500], 'max_depth': [5, 10, None], 'min_samples_leaf': [1, 3, 5]},
    cv=kf, scoring='r2', n_jobs=-1
)
rf_grid.fit(X_train, y_train)
cv_r2 = cross_val_score(rf_grid.best_estimator_, X_train, y_train, cv=kf, scoring='r2')
results.append(report('Random Forest', y_test, rf_grid.predict(X_test), cv_r2))
print(f'Best params: {rf_grid.best_params_}')
print(results[-1])

### Model 5: XGBoost (tuned)

In [ ]:
xgb_grid = GridSearchCV(
    xgb.XGBRegressor(random_state=42, objective='reg:squarederror', verbosity=0),
    {'n_estimators': [200, 500], 'max_depth': [3, 5, 7],
     'learning_rate': [0.03, 0.05, 0.1], 'subsample': [0.8, 1.0]},
    cv=kf, scoring='r2', n_jobs=-1
)
xgb_grid.fit(X_train, y_train)
cv_r2 = cross_val_score(xgb_grid.best_estimator_, X_train, y_train, cv=kf, scoring='r2')
results.append(report('XGBoost', y_test, xgb_grid.predict(X_test), cv_r2))
print(f'Best params: {xgb_grid.best_params_}')
print(results[-1])

### 7.1 Comparison Table

In [ ]:
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

**Reading the table.** XGBoost achieves the best cross-validated R² (≈ 0.55), marginally ahead of Lasso (0.54) and Random Forest (0.54). Linear regression has the best Test R² but the worst CV R² — a clear sign that its test-set advantage was driven by lucky split variance, not genuine generalization. **We treat XGBoost as the primary model** for downstream analysis because:

1. CV R² is the methodologically correct selection metric (test set is too small to compare on, n ≈ 70)
2. The model captures non-linearities (max-contract ceilings, rookie-scale floors, the Age² aging curve)
3. SHAP-based interpretability gives us per-prediction explanations, critical for the "why is this player overpaid?" use case

The narrow gap between models (CV R² of 0.51 to 0.55) is itself the headline finding: **roughly 50% of NBA salary variance is explainable from one season of stats; the other 50% is structural** (CBA-induced).


In [ ]:
best_model = xgb_grid.best_estimator_
y_pred = best_model.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
y_test_d = np.exp(y_test) / 1e6
y_pred_d = np.exp(y_pred) / 1e6
axes[0].scatter(y_test_d, y_pred_d, alpha=0.6, color='#1d428a', edgecolor='white', s=60)
axes[0].plot([y_test_d.min(), y_test_d.max()], [y_test_d.min(), y_test_d.max()], 'r--', label='Perfect prediction')
axes[0].set_xlabel('Actual Salary ($M)')
axes[0].set_ylabel('Predicted Salary ($M)')
axes[0].set_title(f'XGBoost: Predicted vs Actual (Test, R²={r2_score(y_test, y_pred):.3f})')
axes[0].legend()

residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.6, color='#c8102e', edgecolor='white', s=60)
axes[1].axhline(0, color='black', linestyle='--')
axes[1].set_xlabel('Predicted log(Salary)')
axes[1].set_ylabel('Residual (log scale)')
axes[1].set_title('Residuals — heteroscedastic, with bias toward overprediction at low end')

plt.tight_layout()
plt.show()

## 8. Interpretability: SHAP Analysis

Linear coefficients give an aggregate view but mask non-linear interactions. SHAP (SHapley Additive exPlanations) decomposes each prediction into per-feature contributions, giving us both global feature importance and the ability to explain *why this specific player* received this specific predicted salary — exactly what an analyst would want when justifying a recommendation.


In [ ]:
best_model.fit(X_train, y_train)
explainer = shap.TreeExplainer(best_model)
shap_values_full = explainer.shap_values(X)

fig, ax = plt.subplots(figsize=(10, 7))
shap.summary_plot(shap_values_full, X, plot_type='bar', show=False, max_display=15)
plt.title('Global Feature Importance (mean |SHAP value|)')
plt.tight_layout()
plt.show()

In [ ]:
shap.summary_plot(shap_values_full, X, show=False, max_display=15)
plt.tight_layout()
plt.show()

**Interpretation.** Minutes per game and Age dominate global importance, with PTS_per36 and shot volume close behind. The beeswarm shows directionality:
- High MP_pg → strongly positive SHAP (starters earn more, as expected)
- High Age → positive SHAP up to mid-30s, then negative (the aging curve)
- High `is_allstar` → large positive contribution, validating the binary as a CBA-tier proxy (max-contract eligibility)
- Higher PTS_per36 → positive SHAP, but with diminishing returns near the top (max-contract ceiling)


## 9. The Money Shot: Most Over- and Underpaid Players

This is where the model becomes useful as a *decision-making tool*. **A critical methodological choice:** for residual analysis we use **out-of-fold (OOF) predictions** rather than in-sample predictions. With OOF, every player is predicted by a model that did not see them during training — giving honest residuals that reflect genuine inefficiency, not memorization.

This matters because XGBoost can closely fit its training data; using in-sample predictions would compress every residual toward zero and destroy the inefficiency signal we're trying to surface. OOF is the standard correction.

Critically, "overpaid" and "underpaid" here mean *relative to a production-only baseline*, not in any moral or absolute sense. An "underpaid" rookie is on a CBA-mandated rookie-scale contract — they're not a market failure, they're a market structure. But that structure is exactly what creates the trade-deadline opportunity: rookie-scale players with star-level production are the most valuable assets in the league.


In [ ]:
# Out-of-fold predictions: each player predicted by a model trained without them
oof_log = cross_val_predict(best_model, X, y, cv=kf)
df_model['predicted_log'] = oof_log
df_model['predicted_salary'] = np.exp(oof_log)
df_model['residual_dollars'] = df_model['2025-26'] - df_model['predicted_salary']
df_model['residual_pct'] = df_model['residual_dollars'] / df_model['predicted_salary'] * 100

show_cols = ['Player', 'Team', 'Pos', 'Age', 'PTS_per36', 'AST_per36', 'TRB_per36',
             'is_allstar', '2025-26', 'predicted_salary', 'residual_dollars']

overpaid = df_model.nlargest(10, 'residual_dollars')[show_cols].copy()
overpaid['2025-26'] = overpaid['2025-26'].apply(lambda x: f'${x/1e6:.1f}M')
overpaid['predicted_salary'] = overpaid['predicted_salary'].apply(lambda x: f'${x/1e6:.1f}M')
overpaid['residual_dollars'] = overpaid['residual_dollars'].apply(lambda x: f'+${x/1e6:.1f}M')
print('TOP 10 MOST OVERPAID (out-of-fold residuals)')
print(overpaid.to_string(index=False))

In [ ]:
underpaid = df_model.nsmallest(10, 'residual_dollars')[show_cols].copy()
underpaid['2025-26'] = underpaid['2025-26'].apply(lambda x: f'${x/1e6:.1f}M')
underpaid['predicted_salary'] = underpaid['predicted_salary'].apply(lambda x: f'${x/1e6:.1f}M')
underpaid['residual_dollars'] = underpaid['residual_dollars'].apply(lambda x: f'-${abs(x)/1e6:.1f}M')
print('TOP 10 MOST UNDERPAID (out-of-fold residuals)')
print(underpaid.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
over_plot = df_model.nlargest(10, 'residual_dollars').sort_values('residual_dollars')
axes[0].barh(over_plot['Player'], over_plot['residual_dollars'] / 1e6, color='#c8102e', edgecolor='white')
axes[0].set_xlabel('Overpayment ($M, actual − predicted)')
axes[0].set_title('Top 10 Most Overpaid')
axes[0].invert_yaxis()

under_plot = df_model.nsmallest(10, 'residual_dollars').sort_values('residual_dollars', ascending=False)
axes[1].barh(under_plot['Player'], under_plot['residual_dollars'] / 1e6, color='#1d428a', edgecolor='white')
axes[1].set_xlabel('Underpayment ($M, actual − predicted)')
axes[1].set_title('Top 10 Most Underpaid')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

**Reading the leaderboard.**

*Overpaid:* The model flags **Zach LaVine** (~+\$30M over predicted), **Jalen Suggs**, **LaMelo Ball**, **Jalen Green**, and **Paul George** as paid well above their per-36 production. These cluster into two patterns:
1. **Recent rookie-scale extensions paid for expected breakout** — Suggs (\$35M/yr), LaMelo, Jalen Green were all extended off rookie scale at prices anticipating star-level production that hasn't (yet) fully materialized
2. **Declining veterans on legacy max contracts** — Paul George, Khris Middleton, with production slipping below their fresh max prices

The "overpaid" list reads like a list of NBA Twitter trade-discussion targets — a reassuring sanity check that the model is flagging reasonable cases.

*Underpaid:* The list is dominated by:
1. **Ascendant rookie-scale players** — Jalen Williams, Deni Avdija, Austin Reaves are producing at All-Star tier on team-friendly mid-tier deals
2. **Established stars whose contracts pre-date their leap** — **Shai Gilgeous-Alexander** is the best example: his current contract was negotiated before his MVP ascent, leaving roughly \$12M/yr of surplus value
3. **Mid-career veterans on minimum or short-term deals** — players accepting below-market pay for ring chases or one-year prove-it contracts

**For a front office:** A trade target list filtered to underpaid + young + healthy + on movable contract gives you the league's most valuable acquisition targets.


### 9.1 Per-player explanation: why is SGA "underpaid"?

SHAP lets us decompose any individual prediction. Below: the SHAP waterfall for Shai Gilgeous-Alexander, showing exactly which features push his predicted salary up and which pull it down.


In [ ]:
target_player = 'Shai Gilgeous-Alexander'
if target_player in df_model['Player'].values:
    idx = df_model.index[df_model['Player'] == target_player][0]
    pos_in_X = list(X.index).index(idx)
    shap.force_plot(explainer.expected_value, shap_values_full[pos_in_X],
                    X.iloc[pos_in_X], matplotlib=True, show=False)
    plt.title(f'{target_player}: SHAP feature contributions to predicted log-salary')
    plt.tight_layout()
    plt.show()
    actual = df_model.loc[idx, '2025-26']
    pred = df_model.loc[idx, 'predicted_salary']
    print(f'Actual: ${actual/1e6:.1f}M | Predicted (fair-market, OOF): ${pred/1e6:.1f}M | '
          f'Underpaid by ${(pred-actual)/1e6:.1f}M')

## 10. Discussion & Limitations

### What the model gets right
- **Recovers the NBA aging curve.** Age and Age² together capture the well-known late-20s peak earnings window
- **Validates known CBA structures.** The "underpaid" list aligns with rookie-scale cap effects and pre-extension star contracts; the "overpaid" list aligns with declining max-contract veterans and overpriced rookie-scale extensions
- **Reasonable predictive power for a structurally-noisy target.** ~55% CV R² is consistent with what published NBA salary research finds

### Honest limitations

1. **One season of data.** The salary signal is most strongly determined by a player's *recent multi-year* production trend, not just last season. A 3-year rolling average of stats would likely improve fit substantially, but the data engineering is more involved.
2. **No contract-context features.** The model doesn't know whether a player is on a rookie deal, an extension, a max, or a vet-min. Adding contract type as a feature would essentially "predict the CBA away" — interesting if the goal is pure production-to-pay mapping, but then we lose the inefficiency-detection use case.
3. **No defensive impact metrics.** Steals and blocks are crude defensive proxies. Adding Defensive Win Shares, on/off court ratings, or RAPM would likely improve fit for defensive specialists who are systematically misvalued by counting-stat models.
4. **Sample size (n=344).** Tight for tree-based models with 23 features. CV R² standard deviation of ±0.12 reflects this — multi-season data (~1500–2000 player-seasons) would tighten estimates substantially.
5. **Selection bias from the \$2.3M cutoff.** We deliberately excluded league-minimum players. The model cannot predict salaries for that tier, which is fine for the use case (front offices don't trade for vet-mins) but worth flagging.
6. **Survivorship.** The dataset only includes players who appeared in the 2024-25 season; injured-out-the-year stars and recent retirees are missing.

### Is this useful in practice?

**Yes, with appropriate framing.** As a production-only fair-market baseline, the model:
- Generates a defensible per-player "production-implied salary" estimate
- Surfaces market inefficiencies worth investigating further
- Provides a starting point for trade evaluation and contract negotiation

**No, if treated as a literal salary predictor.** The 47% MAPE means individual predictions can be off by half. The right framing is: "the model identifies *which players to investigate further*, not *what to pay them*."


## 11. Deployment

The model is wrapped in an interactive Streamlit dashboard with three tabs:

1. **League Leaderboard** — sortable, filterable table of all 344 players ranked by overpaid/underpaid residual, with team filtering
2. **Player Predictor** — input stats (or pick an existing player), get a fair-market salary estimate with a SHAP waterfall explanation
3. **Model Diagnostics** — feature importance plots and model performance summary

A live deployment link and demo recording are included in the project repo.


In [ ]:
import pickle
with open('model_artifacts.pkl', 'wb') as f:
    pickle.dump({
        'model': best_model,
        'feature_cols': feature_cols,
        'df_model': df_model,
        'X': X,
        'shap_values': shap_values_full,
        'explainer_expected_value': explainer.expected_value,
        'results_df': results_df,
    }, f)
print('Saved model_artifacts.pkl for dashboard.')

## 12. Conclusion

We built and compared five regression methods for predicting NBA player salaries from on-court production. **XGBoost won on cross-validated R² (≈0.55)**, but the narrow gap between all models is itself the headline: about half of NBA salary variance is structural (CBA, contract tier, length-of-service) rather than performance-driven. By reframing the model as a *fair-market value detector* and using out-of-fold residuals, the residuals — not the predictions — become the deliverable, surfacing the league's most over- and underpaid contracts and giving front offices a defensible filter for trade evaluation.

The final tool combines a tuned XGBoost model, SHAP explanations, and a Streamlit dashboard, providing an end-to-end pipeline from raw Basketball-Reference data to an interactive analyst-facing application.
